In [1]:
from google.colab import files

uploaded = files.upload()


Saving stopping_distance_60_values.csv to stopping_distance_60_values (1).csv


In [2]:
import numpy as np
import pandas as pd

df = pd.read_csv("stopping_distance_60_values.csv")
raw_columns = list(df.columns)
df.columns = [
    "obs",
    "v_kmh",
    "v_ms",
    "d_theo",
    "grade_factor",
    "d_sim",
    "sigma",
    "split",
]
df.head()


,obs,v_kmh,v_ms,d_theo,grade_factor,d_sim,sigma,split
0,1,20.000000,5.555556,6.136173,1.000000,6.898341,0.344917,train
1,2,21.694915,6.026365,6.862775,1.000339,7.034415,0.351721,train
2,3,23.389831,6.497175,7.621656,1.000678,7.349341,0.367467,train
3,4,25.084746,6.967985,8.412817,1.001017,8.589261,0.429463,train
4,5,26.779661,7.438795,9.236257,1.001356,9.072780,0.453639,train


In [4]:
failed = []
total = 0


def check(name, ok):
  global total
  total = total + 1
  if ok:
    print("PASSED:", name)
  else:
    print("FAILED:", name)
    failed.append(name)


g = 9.81
# Noise relative to grade-adjusted theoretical stopping distance
d_expected = df.d_theo * df.grade_factor
noise = df.d_sim / d_expected - 1

# Linear fit to d_theo / v = t_r + v / (2 * mu * g) to verify underlying parameters
slope, intercept = np.polyfit(df.v_ms, df.d_theo / df.v_ms, 1)
t_r_fit = intercept
mu_fit = 1.0 / (2.0 * g * slope)

# 1. Speed range and unit conversion
check(
    "speed runs from 20 to 120 km/h",
    abs(df.v_kmh.min() - 20) < 1e-5 and abs(df.v_kmh.max() - 120) < 1e-5,
)
check(
    "speed steps are equal",
    np.diff(df.v_kmh).max() - np.diff(df.v_kmh).min() < 1e-5,
)
check("m/s = km/h / 3.6", np.abs(df.v_ms - (df.v_kmh / 3.6)).max() < 1e-5)

# 2. Distance, grade factor, and uncertainty
check(
    "all values positive",
    (df.v_ms > 0).all()
    and (df.d_theo > 0).all()
    and (df.d_sim > 0).all()
    and (df.sigma > 0).all(),
)
check(
    "uncertainty = 5% of simulated distance",
    np.abs(df.sigma - 0.05 * df.d_sim).max() < 1e-5,
)
check(
    "theoretical distance increases with speed", (np.diff(df.d_theo) > 0).all()
)
check(
    "road grade factor runs from 1.00 to 1.02",
    abs(df.grade_factor.min() - 1.00) < 1e-5
    and abs(df.grade_factor.max() - 1.02) < 1e-5,
)

# 3. Noise properties
check(
    "noise is about 5% (rms between 2.5% and 8%)",
    0.025 < np.sqrt(np.mean(noise**2)) < 0.08,
)
check("no extreme outliers (all within 25%)", np.abs(noise).max() < 0.25)

# 4. Physics recovery
check("theoretical reaction time t_r is 0.7 s", abs(t_r_fit - 0.7) < 1e-4)
check("theoretical friction mu is 0.7", abs(mu_fit - 0.7) < 1e-4)

# 5. Train / test split (60% slow train, 40% fast test for Rubric D6)
check("labels are only train and test", set(df.split) == {"train", "test"})
check(
    "36 train rows and 24 test rows",
    (df.split == "train").sum() == 36 and (df.split == "test").sum() == 24,
)
check(
    "train speeds come before test speeds",
    df.v_ms[df.split == "train"].max() < df.v_ms[df.split == "test"].min(),
)

print()
print(total - len(failed), "out of", total, "checks passed")
if len(failed) > 0:
  print("Failed checks:", failed)


PASSED: speed runs from 20 to 120 km/h
PASSED: speed steps are equal
PASSED: m/s = km/h / 3.6
PASSED: all values positive
PASSED: uncertainty = 5% of simulated distance
PASSED: theoretical distance increases with speed
PASSED: road grade factor runs from 1.00 to 1.02
PASSED: noise is about 5% (rms between 2.5% and 8%)
PASSED: no extreme outliers (all within 25%)
PASSED: theoretical reaction time t_r is 0.7 s
PASSED: theoretical friction mu is 0.7
PASSED: labels are only train and test
PASSED: 36 train rows and 24 test rows
PASSED: train speeds come before test speeds

14 out of 14 checks passed
